<h1>🎛️ Biofilter — Report: <code>aggregate_cohort_variants</code></h1>

Your cohort's variants, matched against the bundle and aggregated into
biological bins.

Three stages: **read** your file, **match** it to the bundle, **aggregate**
the rare variants into bins. `output_grain` decides where you stop.

Sections 4 and 6 are the ones to read — they are the two ways this report
can hand you an empty answer that means something.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

BUNDLE = None
REPORT = "aggregate_cohort_variants"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(bf.core.db_uri)

### 2. A cohort to work with

This notebook has to be runnable, so it writes a small synthetic cohort
from variants the bundle actually carries. Point `COHORT` at your own VCF
and everything below works unchanged.

Two things are deliberate: 200 samples, and two variants on a chromosome
the bundle does not carry.

In [ ]:
import random

from biofilter.modules.report import Bundle

COHORT = OUTPUT_DIR / "demo_cohort.vcf"
PHENOTYPE = OUTPUT_DIR / "demo_phenotype.csv"
N_SAMPLES = 200

with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    rows = bundle.con.execute("""
        SELECT v.chromosome, v.position, v.reference_allele, v.alternate_allele
        FROM variant_masters v
        JOIN entity_locations l ON l.build = 38 AND l.chromosome = v.chromosome
         AND v.position BETWEEN l.start_pos AND l.end_pos
        JOIN gene_masters gm ON gm.entity_id = l.entity_id
        WHERE gm.symbol IN ('CHEK2', 'SMARCB1', 'NF2')
          AND length(v.reference_allele) = 1 AND length(v.alternate_allele) = 1
        ORDER BY v.position LIMIT 600
    """).fetchall()

random.seed(11)
samples = [f"S{i + 1}" for i in range(N_SAMPLES)]
lines = ["##fileformat=VCFv4.2", "##contig=<ID=chr22>"]
lines.append("\t".join(
    ["#CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"] + samples))

for i, (chrom, pos, ref, alt) in enumerate(rows):
    genotypes = ["0/0"] * N_SAMPLES
    if i % 3 == 0:                       # rare: one to three carriers
        for j in random.sample(range(N_SAMPLES), random.choice([1, 2, 3])):
            genotypes[j] = "0/1"
    elif i % 3 == 1:                     # common: about 15%
        genotypes = [f"{int(random.random() < 0.15)}/{int(random.random() < 0.15)}"
                     for _ in samples]
    lines.append("\t".join(
        [f"chr{chrom}", str(pos), f"v{i}", ref, alt, ".", "PASS", ".", "GT"] + genotypes))

# On a chromosome this bundle does not carry — section 4 is about these.
lines.append("\t".join(
    ["chr1", "12010", "offtarget", "A", "C", ".", "PASS", ".", "GT"] + ["0/1"] * N_SAMPLES))

COHORT.write_text("\n".join(lines) + "\n")
PHENOTYPE.write_text("\n".join(
    ["SampleID,Phenotype"]
    + [f"{s},{1 if i < N_SAMPLES // 2 else 0}" for i, s in enumerate(samples)]) + "\n")

print(f"{len(rows)} variants, {N_SAMPLES} samples")
print(f"smallest frequency this cohort can observe: {1 / (2 * N_SAMPLES):.4f}")

### 3. Stage 2 — which of your variants does Biofilter know?

`output_grain="variants"` is what `variant_list_intersect` did. Five
statuses, and the differences between them are the point.

In [ ]:
matched = bf.report.run(REPORT, cohort_file=str(COHORT),
                        phenotype_file=str(PHENOTYPE))
df = matched.to_pandas()

print(f"{len(df):,} cohort variants")
df.groupby("match_status").size().to_frame("variants")

In [ ]:
df[df.match_status == "matched"].head(5)[
    ["cohort_variant_id", "chromosome", "position", "variant_key", "plink_id",
     "gene_symbols", "maf_overall", "maf_case", "maf_control", "is_rare"]
]

`in_bundle_no_gene` and `not_in_bundle` look alike in a spreadsheet
and mean opposite things: the first is a variant Biofilter knows that no
gene contains, the second one it has never heard of. The `note` column
spells out which happened.

### 4. The guard: can the bundle place what you brought?

A cohort file spans the genome. A bundle need not. **Binning a
whole-genome VCF against a single-chromosome bundle does not fail** — it
returns bins for that chromosome and stays silent about the rest.

That is the single most dangerous thing this report could do, so every
run measures it.

In [ ]:
matched.provenance["chromosome_coverage"]

In [ ]:
# When the result leaves your screen, make it a refusal instead.
try:
    bf.report.run(REPORT, cohort_file=str(COHORT), require_full_coverage=True)
except ValueError as exc:
    print("refused:", exc)

### 5. Stage 3 — bins

`output_grain="bins"` is what `variant_binning` did. One row is one
sample in one bin, and only carriers appear.

In [ ]:
bins = bf.report.run(REPORT, cohort_file=str(COHORT),
                     phenotype_file=str(PHENOTYPE),
                     output_grain="bins", group_by="gene",
                     maf_cutoff=0.01)
bdf = bins.to_pandas()

print(f"{len(bdf):,} (sample, bin) rows across {bdf.bin_name.nunique()} bins")
bdf.head(5)

In [ ]:
# The burden, which is what a downstream test consumes.
bdf.groupby(["bin_name", "sample_class"]).agg(
    samples=("sample", "nunique"),
    alt_alleles=("alt_count", "sum"),
    variants=("variant_count", "sum"),
)

### 6. The other empty result, and it is arithmetic

With **N samples the smallest observable minor allele frequency is
1/(2N)** — one allele copy in one person. A 20-sample cohort cannot see
anything rarer than 0.025, so asking it for `maf_cutoff=0.01` keeps only
variants nobody carries and every bin comes back empty.

Correct, and invisible in the rows. So the report says it.

In [ ]:
bins.provenance["rare_variants"]

In [ ]:
# The same cohort cut down to 10 samples, asking for something it
# cannot observe.
small = OUTPUT_DIR / "demo_small.vcf"
head, *body = COHORT.read_text().splitlines()
cols = body[0].split("\t")
small.write_text("\n".join(
    [head, "\t".join(cols[:9] + cols[9:19])]
    + ["\t".join(r.split("\t")[:9] + r.split("\t")[9:19]) for r in body[1:]]) + "\n")

tiny = bf.report.run(REPORT, cohort_file=str(small), output_grain="bins",
                     maf_cutoff=0.01)
print("rows:", len(tiny.to_pandas()))
print(tiny.provenance["rare_variants"]["means"])

### 7. Four kinds of bin, and how far each reaches

Stage 2 is positional, and only 39,306 of the bundle's 72,660 genes carry
build-38 coordinates — so half the catalogue is out of reach whatever the
grouping.

In [ ]:
for group_by in ("gene", "gene_group", "locus_type", "pathway"):
    out = bf.report.run(REPORT, cohort_file=str(COHORT),
                        phenotype_file=str(PHENOTYPE),
                        output_grain="bins", group_by=group_by, maf_cutoff=0.01)
    frame = out.to_pandas()
    reach = out.provenance["bin_coverage"]
    print(f"  {group_by:<11} {len(frame):>6,} rows, "
          f"{frame.bin_name.nunique() if len(frame) else 0:>4} bins   "
          f"reaches {reach['genes_this_bin_type_can_reach']:,} genes "
          f"({reach['share']:.1%})")

### 8. Which frequency the rare filter uses

With both arms present the default filters on the **larger** of the case
and control MAFs — BioBin's rule, so a variant common in cases and absent
in controls is not swept into a rare bin.

In [ ]:
for label, params in [
    ("case/control (default)", {"rare_case_control": True}),
    ("control only", {"rare_case_control": False, "overall_major_allele": False}),
    ("overall", {"rare_case_control": False, "overall_major_allele": True}),
]:
    out = bf.report.run(REPORT, cohort_file=str(COHORT),
                        phenotype_file=str(PHENOTYPE),
                        output_grain="bins", maf_cutoff=0.01, **params)
    rare = out.provenance["rare_variants"]
    print(f"  {label:<24} {rare['rare']:>4} rare, "
          f"{rare['rare_with_carriers']:>4} with carriers")

### 9. One table that travels, one file that does not

`variant_to_bin` — what each bin is made of — is a **second table on the
result**, always there when you ask for bins. Two runs differing only in
`maf_cutoff` produce different bins and the main table does not say which
variants moved; this is how you find out. Being a table rather than a
file, it cannot be lost or go stale without anything noticing.

`plink_extract_path` writes a real file, because PLINK reads files. It
matches on the id in your `.bim`, not on coordinates, so the id your own
file used is preferred — a list of `chr:pos` strings extracts nothing
from a dataset keyed by rsIDs.

In [ ]:
audited = bf.report.run(
    REPORT, cohort_file=str(COHORT), phenotype_file=str(PHENOTYPE),
    output_grain="bins", maf_cutoff=0.01,
)

print("tables on the result:")
for name, table in audited.tables.items():
    print(f"  {name:<16} {table.num_rows:>6,} rows")

display(audited.extra_tables["variant_to_bin"].to_pandas().head(5))

keep = bf.report.run(
    REPORT, cohort_file=str(COHORT),
    plink_extract_path=str(OUTPUT_DIR / "keep.txt"),
)
for artifact in keep.artifacts:
    print(f"\nfile: {artifact.name} — {artifact.description}")

### 10. What the run wants you to know

This report proceeds through three situations that can make its answer
misleading rather than refusing outright. Each is logged when it happens
**and** recorded in the provenance, because whoever opens the result next
month does not have the log.

In [ ]:
small = bf.report.run(REPORT, cohort_file=str(COHORT),
                     output_grain="bins", maf_cutoff=0.0001)

for warning in small.provenance["warnings"]:
    print("⚠️ ", warning["message"])

print("\nno warnings on a clean run:",
      bf.report.run(REPORT, cohort_file=str(COHORT),
                    output_grain="variants").provenance["warnings"])

### 11. Export

In [ ]:
for path in bins.write(OUTPUT_DIR / "aggregate_cohort_variants.csv"):
    print(path)

### 12. The same thing on the command line

```bash
biofilter report run --report-name aggregate_cohort_variants \\
    --param cohort_file=./cohort.vcf.gz \\
    --param phenotype_file=./phenotype.csv \\
    --param output_grain=bins \\
    --param group_by=gene \\
    --param maf_cutoff=0.01 \\
    --param require_full_coverage=true \\
    --output bins.csv
```